In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [116]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])

In [117]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [118]:
print("Total rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())

Total rows: 57671
Duplicate rows: 0


In [119]:
df.isnull().sum()

,0
text,0
emotion,0


In [120]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emotion in unique_emotions:
    emotion_numbers[emotion] = i
    i += 1
df['emotion'] = df['emotion'].map(emotion_numbers)

In [121]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [122]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [123]:
import string

def remove_punc(text):
    for punc in string.punctuation:
        text = text.replace(punc, '')
    return text

In [124]:
df['text'] = df['text'].apply(remove_punc)

In [125]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [126]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [127]:
import nltk

In [128]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [129]:
# Downloading all required nltk resources together (was previously split
# across two cells, with punkt_tab only fetched right before it was needed —
# consolidated here so all downloads happen up front).
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [130]:
stop_words = set(stopwords.words('english'))

In [131]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [132]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if i not in stop_words:
            cleaned.append(i)
    return " ".join(cleaned)

In [133]:
df['text'] = df['text'].apply(remove)

In [134]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
57666,feel wonderful today,5
57667,mean everything deeply,2
57668,unexpected suddenly,3
57669,feel wonderful,5


In [135]:
print("Total:", len(df))
print(df["emotion"].value_counts())
print("Duplicate text:", df["text"].duplicated().sum())
print("Duplicate complete rows:", df.duplicated().sum())

Total: 57671
emotion
1    12606
4    11832
2     9500
0     7965
3     7918
5     7850
Name: count, dtype: int64
Duplicate text: 3288
Duplicate complete rows: 3278


In [136]:
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print("Rows after exact-duplicate removal:", len(df))
print("Unique texts:", df['text'].nunique())

Rows after exact-duplicate removal: 54383
Unique texts: 54383


In [137]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['emotion'],
    test_size=0.20,
    random_state=42,
    stratify=df['emotion']
)

In [138]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [139]:
bow_vectorizer = CountVectorizer()

In [140]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [141]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [142]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [143]:
pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))
print(confusion_matrix(y_test, pred_bow))
print(classification_report(y_test, pred_bow))

0.9051208973062426
[[1446   19   12    4   12   61]
 [ 137 2036    5    1   21  110]
 [  61    8 1670    0    2  135]
 [  37    0    3 1483   11   43]
 [ 107   11    9    8 1775   99]
 [  63    7   34    4    8 1435]]
              precision    recall  f1-score   support

           0       0.78      0.93      0.85      1554
           1       0.98      0.88      0.93      2310
           2       0.96      0.89      0.93      1876
           3       0.99      0.94      0.96      1577
           4       0.97      0.88      0.92      2009
           5       0.76      0.93      0.84      1551

    accuracy                           0.91     10877
   macro avg       0.91      0.91      0.90     10877
weighted avg       0.92      0.91      0.91     10877



In [144]:
tfid_vectorizer = TfidfVectorizer()
X_train_tfid = tfid_vectorizer.fit_transform(X_train)
X_test_tfid = tfid_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfid, y_train)

MultinomialNB()

In [145]:
pred_tfid = nb2_model.predict(X_test_tfid)
print(accuracy_score(y_test, pred_tfid))
print(confusion_matrix(y_test, pred_tfid))
print(classification_report(y_test, pred_tfid))

0.9107290613220557
[[1426   29   12    3   18   66]
 [ 103 2086    7    0   24   90]
 [  47   12 1682    1    4  130]
 [  33    0    4 1483   16   41]
 [  79   20   12    8 1803   87]
 [  61   11   41    1   11 1426]]
              precision    recall  f1-score   support

           0       0.82      0.92      0.86      1554
           1       0.97      0.90      0.93      2310
           2       0.96      0.90      0.93      1876
           3       0.99      0.94      0.97      1577
           4       0.96      0.90      0.93      2009
           5       0.78      0.92      0.84      1551

    accuracy                           0.91     10877
   macro avg       0.91      0.91      0.91     10877
weighted avg       0.92      0.91      0.91     10877



In [146]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression(max_iter=1000)

In [147]:
logistic_model.fit(X_train_bow, y_train)

LogisticRegression(max_iter=1000)

In [148]:
log_pred = logistic_model.predict(X_test_bow)
print(accuracy_score(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))
print(classification_report(y_test, log_pred))

0.958904109589041
[[1469   26    7    5   19   28]
 [  30 2241    3    1   21   14]
 [  11    2 1796    0    4   63]
 [   2    2    0 1542   19   12]
 [  23   14    5   17 1933   17]
 [  31    7   51    6    7 1449]]
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1554
           1       0.98      0.97      0.97      2310
           2       0.96      0.96      0.96      1876
           3       0.98      0.98      0.98      1577
           4       0.97      0.96      0.96      2009
           5       0.92      0.93      0.92      1551

    accuracy                           0.96     10877
   macro avg       0.96      0.96      0.96     10877
weighted avg       0.96      0.96      0.96     10877



In [149]:
import joblib
joblib.dump(bow_vectorizer, "bow_vectorizer.pkl")
joblib.dump(logistic_model, "emotion_model.pkl")
joblib.dump(emotion_numbers, "emotion_numbers.pkl")

['emotion_numbers.pkl']

In [150]:
print(emotion_numbers)
print(df['emotion'].value_counts())
print(confusion_matrix(y_test, log_pred))

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}
emotion
1    11551
4    10042
2     9380
3     7884
0     7769
5     7757
Name: count, dtype: int64
[[1469   26    7    5   19   28]
 [  30 2241    3    1   21   14]
 [  11    2 1796    0    4   63]
 [   2    2    0 1542   19   12]
 [  23   14    5   17 1933   17]
 [  31    7   51    6    7 1449]]
